In [10]:
# Install latest version from local files (only necessary after pulling updated version from Git).
# Restart Jupyter kernel when install has finished.
%pip install -e ..

Obtaining file:///home/maarten/Documents/Projects/BioPAP/BioPAP
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for BioPAP (pyproject.toml) ... done
  Created wheel for BioPAP: filename=biopap-0.1.dev25+ga3a7ede54.d20251125-0.editable-py3-none-any.whl size=3384 sha256=7d7146e4aeb14da0544b0a73ad56f48c0560af862ecee150342041ab4e4fb548
  Stored in directory: /tmp/pip-ephem-wheel-cache-drvu0qkw/wheels/a9/55/80/ba98fe05e3d05116737c29a1c667eb5bffec478c135f508c7b
Successfully built BioPAP
  Attempting uninstall: BioPAP
    Found existing installation: BioPAP 0.1.dev25+ga3a7ede54.d20251125
    Uninstalling BioPAP-0.1.dev25+ga3a7ede54.d20251125:
      Successfully uninstalled BioPAP-0.1.dev25+ga3a7ede54.d20251125
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Some Windows installations require explicitly installing mne (required only once; restart Jupyter kernel after installing).
%pip install mne

In [1]:
from biopap.pipeline import PipelineStep, Pipeline

from biopap.modules.load_data import LoadData
from biopap.modules.anonymize import Anonymize
from biopap.modules.rereference import Rereference
from biopap.modules.eog_channels import EOGChannels
from biopap.modules.iir_filter import IIRFilter
from biopap.modules.downsample import Downsample
# from biopap.modules.segmentation import Segmentation

In [2]:
import matplotlib.pyplot as plt
from pathlib import Path
from pprint import pprint

### Define the data and config files

The config file holds the settings for each of the steps of the pipeline we are going to create. They are kept in a separate file to make it easier to use the same settings for multiple runs, for instance when running preprocessing on many files in parallel.

In [3]:
# To avoid issues with local settings and GitHub updates, create a small notebook
# called 'my_config.ipynb' to hold variables that are likely to change locally. 
# In the same folder as this file, there is an example notebook called
# 'example.my_config.ipynb' that can serve as a template.

config_notebook = './my_config.ipynb'

if Path(config_notebook).exists():
    %run {config_notebook}
else:
    data_file = '/biopap/data/example.bdf'
    config_file = './data/pipeline.yaml'

if not Path(config_file).exists:
    print(f"Config file {config_file} does not exist.")

if not Path(data_file).exists:
    print(f"Data file {data_file} does not exist.")

### Loading data

In [4]:
# Read the data file and load it into memory.
load = LoadData(file_path=data_file)

Extracting EDF parameters from /data/BioPAP/data/CBN1BD071212-Cropped.bdf...
BDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 3276799  =      0.000 ...  1600.000 secs...


### Setting up the pipeline

In [5]:
# Create a new pipeline, loading the raw data and the configuration file.
pipe = Pipeline(data=load.raw, config_file=config_file)

In [6]:
# Print the pipeline's config attribute to see the settings you loaded.
pprint(pipe.config)

{'Anonymize': {'daysback': 100,
               'experimenter': 'J. Jansen',
               'participant_id': 'AB00123'},
 'Downsample': {'sfreq': 500},
 'Downsample_1000': {'sfreq': 1000},
 'EOGChannels': {'anode_channels': ['EXG3', 'EXG5'],
                 'cathode_channels': ['EXG4', 'EXG6'],
                 'ref_channels': ['hEOG', 'vEOG']},
 'IIRFilter': {'h_freq': 35, 'l_freq': 0.1},
 'Rereference': {'ref_channels': ['EXG1', 'EXG2']}}


In [7]:
# Print info of the pipeline's raw attribute to see the loaded data's main info.
print(load.raw.info)

<Info | 8 non-empty values
 bads: []
 ch_names: Fp1, AF7, AF3, F1, F3, F5, F7, FT7, FC5, FC3, FC1, C1, C3, C5, ...
 chs: 79 EEG, 1 Stimulus
 custom_ref_applied: False
 highpass: 0.0 Hz
 lowpass: 417.0 Hz
 meas_date: 2012-12-07 12:14:08 UTC
 nchan: 80
 projs: []
 sfreq: 2048.0 Hz
 subject_info: <subject_info | his_id: CBN1BD071212>
>


### Adding steps to the pipeline
A pipeline is made up of a number of steps that are executed in order. Each step's output becomes the next step's input. The first step starts with the raw data loaded when the pipeline was created. You first add all necessary the steps to the pipeline, without actually running any processing. Only when you are satisfied the pipeline is complete, the entire pipleline is run with one command.

A step simply consists of one of the available processing classes. The specific settings for that class are read from the config file when the step is added. When you need to run the same class twice with different settings, you can use a custom label in the configuration file to keep the different sets of settings apart.

When adding a step, you can tell the pipeline to plot the resulting (processed) data right after that step has finished. You can also tell the pipeline to pause after a step.

To change the pipeline, call the reset method, and re-add the steps in the right order. There is currently no way to change the order of steps in an existing pipeline, or to remove individual steps.

In [ ]:
# Add processing steps in the order they should be run.
pipe.add_step(Anonymize)
pipe.add_step(Downsample, plot=True, pause=True)
pipe.add_step(IIRFilter)

# print pipeline overview, including each step's configuration settings
print(*pipe.steps, sep='\n')

In [ ]:
pipe.run()

In [ ]:
# When adding multiple steps for same class w/ different settings, use config label to keep them apart.
pipe.add_step(Downsample, cfg_label='Downsample_1000', plot=True)

print(*pipe.steps, sep='\n')

In [ ]:
# Settings from the config file can also be overridden ad hoc
pipe.add_step(IIRFilter, cfg={'h_freq': 50})

print(*pipe.steps, sep='\n')

In [ ]:
# Call reset to clear the entire pipeline.
pipe.reset()

In [8]:
# Adding steps.
pipe.add_step(Anonymize)
pipe.add_step(Rereference)
pipe.add_step(EOGChannels)
pipe.add_step(IIRFilter)
pipe.add_step(Downsample)

print(*pipe.steps, sep='\n')

0: Anonymize ← {'daysback': 100, 'experimenter': 'J. Jansen', 'participant_id': 'AB00123'}; not plotting
1: Rereference ← {'ref_channels': ['EXG1', 'EXG2']}; not plotting
2: EOGChannels ← {'ref_channels': ['hEOG', 'vEOG'], 'anode_channels': ['EXG3', 'EXG5'], 'cathode_channels': ['EXG4', 'EXG6']}; not plotting
3: IIRFilter ← {'l_freq': 0.1, 'h_freq': 35}; not plotting
4: Downsample ← {'sfreq': 500}; not plotting


In [9]:
# Before running the pipeline, test if the configuration is correct (optional).
cfg_correct = pipe.test_config()

No errors


### Running the pipeline

In [ ]:
# Run the entire pipeline and plot the end results.
pipe.run(plot=True)